# Assigment 2 - Python dictionaries and file I/O

### Overview
------------

Most FASTA files you will encounter will contain multiple sequences. Further, for genes you will often get 3' and 5' untranslated sequences (UTRs) flanking the coding sequence. In this exercise you will be writing a function for parsing a multiple-sequence FASTA file and translating each sequence, locating the coding sequence in between the UTRs. Finally, you will look at the prevalence of amino acids across the different sequences.

Biological Learning Objectives

- Use a codon table and DNA sequences to identify and translate coding sequences
- Count and compare amino acid usage for the population of sequences 

Computational Learning Objectives

- Create and modify a dicitionary
- Use keys to retrieve values stored in a dictionary
- Step through each key and value in a dictionary inside a `for` loop
- Use conditionals to alter the behaviour or a `for` or `while` loop, terminating early if necessary
- Open and read a text file
- Write to a text file

### Instructions
----------------

- Save a copy of this notebook as `~/qbXX-answers/day2-morning/python_dictionaries.ipynb`.
- Copy the files [`sequences.fa`](https://raw.githubusercontent.com/bxlab/cmdb-quantbio/refs/heads/main/assignments/bootcamp/dictionaries_file_io/sequences.fa) and [`codons.tsv`](https://raw.githubusercontent.com/bxlab/cmdb-quantbio/refs/heads/main/assignments/bootcamp/dictionaries_file_io/codons.tsv) into `~/qbXX-answers/day2-morning/` (but do not submit them with your answer)
- Fill in answers in the available code/markdown cells below.
- Remember to comment your code to help yourself and us know what each part is intended to do.

### What to turn in
-------------------

- This filled-in notebook
- Your codon usage file

### Scoring
-----------

- 2.0 pts - Multiple-sequence fasta load function
- 0.5 pts - Codon table load function
- 4.0 pts - Translation function
    - 1.5 pts - Loop stepping through sequence with correct reading frame
    - 1.0 pts - Skipping 5' UTR
    - 0.5 pts - Converting coding sequence to amino acids
    - 1.0 pts - Skipping 3' UTR
- 2.0 pts - Counting amino acid usage
- 0.5 pts - Converting counts to percentages
- 1.0 pts - Writing usage results file

10 pts total

-------------------

1. Start by setting the correct working directory (this is important if you don't want to use full paths for your file names)


In [39]:
%cd ~/qb26-answers/

/Users/cmdb/qb26-answers


/opt/anaconda3/envs/qb26/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/opt/anaconda3/envs/qb26/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


2. Building on the [code](https://github.com/bxlab/cmdb-quantbio/blob/main/lectures/python_dicts_file_io/livecoding.ipynb) for reading in a single FASTA sequence, adapt it to read in multiple sequences from a single FASTA file, storing each sequence in a dictionary using the sequence name as the key. Wrap this code in a function such that it takes a file name as the only function argument and returns the dictionary of sequences.

- To check if a line represents the start of a new sequence, consider using the string method `.startswith()`
- Don't forget to close your filestream

In [40]:
#creates a function that takes a file name as the only function argument
def fasta_read(file_name):

    all_sequences = {}

    for line in open(file_name):
        #capture sequence name wih a conditional
        if line.startswith('>'):
            name = line.rstrip() [1:]
            #create a place to store our sequence each time we encounter a new name
            all_sequences[name] = []
            continue
        #store each line we read into a list
        all_sequences[name].append(line.rstrip("\n"))

        #step through our dictionary of sequences and reformat
    for name, seqs in all_sequences.items():
        #think of what variable contains the list (seqs)
        #replace our list of sequences with a string of sequences
        all_sequences[name]= ''.join(seqs)   

    #return the dictionary of sequences
    return all_sequences

# can use these to check that the function works
#final_out = fasta_read('day2-morning/sequences.fa')

#print(final_out.keys())
#print(final_out['CCDS2.2|Hs110|chr1'])




3. Wrap the code for reading in the codon table into a function, taking a file name in as the argument and returning the dictionary of codon/amino acid pairs.

In [41]:
#creates a function that takes a file name as the only function argument
def read_codons(codon_file):

    #create a dictionary of codon/AA pairs
    codons = {}
    for line in open(codon_file):
        fields = line.strip().split('\t')
        #split always gives us a list
        codons[fields[0]] = fields[1]

    return codons

# can use these to check that the function works
#output = codon_table('day2-morning/codons.tsv')
#print(output)




4. Write a function for translating the CDS sequences into amino acid sequences. This function will need to take in two arguments, the codon table and the DNA sequence. The DNA sequences contain untranslated sequences (UTRs) at the start and end so you will need to step through the sequences to find the first methionine (M). Likewise, you will need to stop when you encounter the first step codon (*) rather than translating through the end of the sequence.

- Using a `while` loop may be useful for this task, but it is not required as `for` loops can also work
- You will need to consider three different parts reading the sequence:
    1. Have you reached the start of the coding sequence
    2. Do you need to record the current codon's amino acid
    3. Have you reached the end of the coding sequence

In [ ]:
# create our function to translate CDS into AA sequence 
# use codon table and the DNA sequence as arguments for the function
def translate(sequence, codon_dict):
    # create an empty list to store the amino acids as we move through a sequence
    AAs = []

    #create a variable to store the start of the coding sequence
    start = sequence.find('ATG')

    #apply starting point to range so we begin counting in the right place
    for pos in range(start, len(sequence), 3):
        codon = sequence[pos:pos +3]

        #record the current codon's amino acid
        #return 0 if there is no codon/key matching the codon
        AAs.append(codon_dict.get(codon, '0'))

        #use conditional to check if the end of the coding sequence has been reached
        #use -1 to look up the last item in the list/amino acid we just added
        if AAs[-1] == "*":
            break

    #return the list of amino acids
    return AAs


5. Finally, put it all together, loading in the FASTA sequences and codon table, and translating the into amino acids. Once you have the amino acid sequences, count the number of times each amino acid is used. Finally, convert these counts into percentages and write them to a tab-separated file with the first column being the amino acid letter and the second column being the percent usage.

- To get the percentages, it will helpful to keep a running total of the number of amino acids as you find the counts
- The dictionary method `.setdefault` may be useful for intializing you count dictionary for each new amino acid
- Don't forget to close your filestream

In [43]:
# load the fasta sequences 
fasta_sequence = fasta_read('day2-morning/sequences.fa')
# load the codon table
codon_table = read_codons('day2-morning/codons.tsv')

# use translate() to translate amino acids
AA_counter = {}
total = 0
# loop through fasta_sequence dictionary to count translate each sequence in the dictionary
for name, sequence in fasta_sequence.items():
    AA_sequence = translate(sequence, codon_table)
    # count the number of times each amino acid is used
    for character in AA_sequence:
        AA_counter.setdefault(character, 0)
        AA_counter[character] += 1
        total += 1

# write into a tab-seperated file with the first column being amino acid letter, and the second column being % usage
fs = open('AA_percent.tsv', mode = 'w')
fs.write(f"Amino Acid\tPercent Usage\n")
for AA, count in AA_counter.items():
    # calculate the amino acid count into percentages
    percent = count / total * 100
    # write the amino acids, and the percents into the new .tsv
    fs.write(f"{AA}\t{percent}\n")

fs.close()
